In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    min as spark_min,
    max as spark_max,
    sum as spark_sum,
    when,
    to_timestamp
)

RAW_PEDIDOS_PATH = "abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/ecommerce_pedidos.csv/"

adls_options = get_adls_options()

df_pedidos_raw = (
    spark.read
    .format("csv")
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "false")
    .option("sep", ",")
    .option("recursiveFileLookup", "true")
    .load(RAW_PEDIDOS_PATH)
)

print("Leitura da Raw ecommerce_pedidos concluída.")
print(f"Total de colunas: {len(df_pedidos_raw.columns)}")

df_pedidos_raw.printSchema()
display(df_pedidos_raw.limit(10))

In [0]:
total_linhas = df_pedidos_raw.count()

print(f"Total de linhas lidas: {total_linhas}")
print("Colunas encontradas:")
print(df_pedidos_raw.columns)

In [0]:
df_pedidos_datas = (
    df_pedidos_raw
    .withColumn("dt_pedido_ts", to_timestamp(col("dt_pedido")))
)

df_resumo_datas = (
    df_pedidos_datas
    .agg(
        spark_min("dt_pedido_ts").alias("data_pedido_mais_antiga"),
        spark_max("dt_pedido_ts").alias("data_pedido_mais_recente"),
        count(when(col("dt_pedido").isNull(), True)).alias("dt_pedido_nula"),
        count(when(col("dt_pedido_ts").isNull() & col("dt_pedido").isNotNull(), True)).alias("dt_pedido_invalida")
    )
)

display(df_resumo_datas)

In [0]:
df_resumo_chave = (
    df_pedidos_raw
    .agg(
        count("*").alias("total_linhas"),
        countDistinct("id_pedido").alias("id_pedido_distintos"),
        count(when(col("id_pedido").isNull(), True)).alias("id_pedido_nulo"),
        count(when(col("id_cliente").isNull(), True)).alias("id_cliente_nulo"),
        count(when(col("id_endereco_entrega").isNull(), True)).alias("id_endereco_entrega_nulo")
    )
)

display(df_resumo_chave)

df_status = (
    df_pedidos_raw
    .groupBy("status_pedido")
    .count()
    .orderBy(col("count").desc())
)

display(df_status)

In [0]:
from pyspark.sql.functions import col, min as spark_min, max as spark_max, countDistinct, count, when

df_pedidos_cliente = (
    df_pedidos_raw
    .withColumn("id_cliente_int", col("id_cliente").cast("int"))
)

df_resumo_clientes_pedidos = (
    df_pedidos_cliente
    .agg(
        spark_min("id_cliente_int").alias("menor_id_cliente"),
        spark_max("id_cliente_int").alias("maior_id_cliente"),
        countDistinct("id_cliente_int").alias("clientes_distintos_com_pedido"),
        count(when(col("id_cliente_int").isNull(), True)).alias("id_cliente_invalido_ou_nulo")
    )
)

display(df_resumo_clientes_pedidos)

In [0]:
display(
    df_pedidos_cliente
    .select("id_cliente", "id_cliente_int")
    .distinct()
    .orderBy(col("id_cliente_int").desc())
    .limit(20)
)

In [0]:
display(
    df_pedidos_cliente
    .filter(col("id_cliente_int") > 10000)
    .select("id_pedido", "id_cliente", "dt_pedido", "status_pedido", "valor_total")
    .orderBy(col("id_cliente_int").desc())
    .limit(50)
)

In [0]:
from pyspark.sql.functions import col

df_clientes_esperados = spark.range(1, 10001).withColumnRenamed("id", "id_cliente_int")

df_clientes_com_pedido = (
    df_pedidos_cliente
    .select("id_cliente_int")
    .distinct()
)

df_clientes_sem_pedido = (
    df_clientes_esperados
    .join(df_clientes_com_pedido, on="id_cliente_int", how="left_anti")
    .orderBy("id_cliente_int")
)

display(df_clientes_sem_pedido)